# Financial Fraud Detection (Pix)

Este projeto simula um motor de detecção de fraudes para transações financeiras móveis (semelhante ao Pix). O objetivo é processar um grande volume de logs transacionais para identificar padrões anômalos e blindar o sistema contra perdas financeiras.

Diferente de datasets didáticos pequenos, este projeto utiliza o **PaySim**, contendo mais de **6 milhões de registros**, exigindo o uso de tecnologias de Big Data (Spark) para processamento distribuído, já que ferramentas tradicionais (Excel/Pandas local) não performam adequadamente nesta escala.

- **Desafio de Negócio:** Detectar a minoria fraudulenta (0.1% dos casos) sem bloquear clientes legítimos (Falsos Positivos), lidando com o severo desbalanceamento de classes.
## 🛠 Tecnologias Utilizadas

* **Linguagem:** Python (PySpark API)
* **Processamento:** Apache Spark (Computação Distribuída & In-Memory Processing)
* **Armazenamento:** Databricks File System (DBFS) e Delta Lake (Camadas Bronze/Silver)
* **Machine Learning:** ?
* **Ambiente:** Databricks Community Edition (Serverless Compute)
## 📂 Dados

O dataset utilizado é o **PaySim: Mobile Money Simulator**, criado a partir de logs reais de transações financeiras anonimizadas.

* **Fonte:** [Kaggle - PaySim Dataset](https://www.kaggle.com/datasets/ealaxi/paysim1)
* **Volume:** ~6.3 milhões de transações (Simulação de Big Data Real)
* **Tamanho:** ~470MB (CSV Bruto)
* **Autor:** Edgar Lopez-Rojas



## Imports, Schema, Bronze -> Silver... Passos Iniciais

In [0]:
# Imports

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set_style("dark")
plt.style.use('dark_background')

In [0]:
# Criando a base de dados "fraud_det_project"

spark.sql("USE CATALOG workspace") 
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_det_project")
spark.sql("USE fraud_det_project")
print("Banco de dados 'fraud_det_project' criado e selecionado!")

In [0]:
# Lendo os dados e criando a tabela Bronze

df_bronze = spark.read.table("workspace.default.paysim_bronze")

print("✅ Tabela encontrada! Contagem de linhas:")
print(f"Total de linhas no dataset: {df_bronze.count()}")


`step` = Unidade de tempo. Hora em que aconteceu a transação. Vai de 1 a 743.

- Vamos particionar pelo `step`

In [0]:
%sql

-- Casting usando SQL e criando a tabela Silver --

CREATE OR REPLACE TABLE paysim_silver
USING DELTA
PARTITIONED BY (step)
AS
SELECT 
    CAST(step AS INT) as step,
    CAST(type AS STRING) as type,
    CAST(amount AS DOUBLE) as amount,
    CAST(nameOrig AS STRING) as name_orig,
    CAST(oldbalanceOrg AS DOUBLE) as old_balance_org,
    CAST(newbalanceOrig AS DOUBLE) as new_balance_orig,
    CAST(nameDest AS STRING) as name_dest,
    CAST(oldbalanceDest AS DOUBLE) as old_balance_dest,
    CAST(newbalanceDest AS DOUBLE) as new_balance_dest,
    CAST(isFraud AS INT) as is_fraud,
    CAST(isFlaggedFraud AS INT) as is_flagged_fraud
    
FROM workspace.default.paysim_bronze;

In [0]:
# Lendo o a tabela Silver

df_silver = spark.read.table("paysim_silver")

### Removendo Duplicatas e Tratando Nulos

In [0]:
# Removendo duplicatas
df_deduplicated = df_silver.dropDuplicates()

# Verificando nulos
null_check = df_deduplicated.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_deduplicated.columns
])

display(null_check)

## FEATURES - ERRO Matemático da Transação e Normalizando `step`

| Feature | O que é? |
| --- | --- |
| **`error_orig`**      | A diferença matemática no saldo da vítima.  |
| **`error_dest`**      | A diferença matemática no saldo do destino. |
| **`hour_of_day`**     | Conversão de `step` para hora (0-23h).      |

In [0]:
# Lista de tipos onde a conta de Origem RECEBE dinheiro
input_type = ["CASH_IN"] 

# Erro da Origem
df_calc = df_deduplicated.withColumn("error_orig", 
    F.when(F.col("type").isin(input_type), 
         F.col("old_balance_org") + F.col("amount") - F.col("new_balance_orig")
    ).otherwise(
         F.col("old_balance_org") - F.col("amount") - F.col("new_balance_orig")
    )
)

# Erro do Destino
df_final = df_calc.withColumn("error_dest", 
    F.when(F.col("name_dest").startswith("M"), 
         F.lit(0.0)  # F.lit para valor literal
    ).when(F.col("type").isin(input_type), 
         F.col("old_balance_dest") - F.col("amount") - F.col("new_balance_dest")
    ).otherwise(
         F.col("old_balance_dest") + F.col("amount") - F.col("new_balance_dest")
    )
)


display(df_final.select("type", "amount", "is_fraud", "error_orig", "error_dest")
                .filter(F.col("step") == 496)
                .limit(5))

In [0]:
# Step -> Hora do dia

df_final = df_final.withColumn("hour_of_day", F.col("step") % 24)

### Interpretando em `error_orig/dest`
| Variável | Sinal | Significado | Grau de Suspeita |
| :--- | :---: | :--- | :--- |
| **error_orig** | **Negativo (-)** | **Dinheiro preso:** O valor deveria ter saído da conta de origem, mas o saldo **não baixou** (ou baixou menos do que devia). |  **FRAUDE CRÍTICA** *(O fraudador transfere/saca sem que o débito ocorra na conta).* |
| **error_orig** | **Positivo (+)** | **Desconto excessivo:** O saldo da origem baixou **mais** do que o valor da transação. |  **Baixa** (Geralmente indica erro de sistema ou cobrança de taxas ocultas). |
| **error_dest** | **Positivo (+)** | **Dinheiro sumiu:** O valor deveria ter entrado na conta destino, mas o saldo **não subiu**. |  **LAVAGEM / CONTA LARANJA** *(Indica conta "fantasma" ou saque imediato para não deixar rastro).* |
| **error_dest** | **Negativo (-)** | **Dinheiro brotou:** O saldo do destino subiu **mais** do que o valor enviado. |  **Baixa** *(Geralmente erro de sincronização do sistema).* |

> Para considerar fraude, o erro deve ser significativo. Erros muito pequenos (ex: `1.16e-10`) devem ser ignorados usando um *threshold* de tolerância (ex: `abs(error) > 0.05`).

## EXPLORATORY DATA ANALYSIS (EDA)

In [0]:
# Total por Tipo de Transação

display(df_final.groupBy("type").count().orderBy("count", ascending = False))

In [0]:
# Média e Mediana
# df_stats = média e mediana por Tipo

df_stats = df_final.groupBy("type").agg(
    F.avg("amount").alias("mean_amount"), 
    F.percentile_approx("amount", 0.5).alias("median_amount")
    ).orderBy("mean_amount", ascending = False)

display(df_stats)

In [0]:
# Contas de Origem com mais fraudes

df_final.filter(F.col("is_fraud") == 1) \
    .groupBy("name_orig") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(5)

# Contas de Destino com mais fraudes

df_final.filter(F.col("is_fraud") == 1) \
    .groupBy("name_dest") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(5)

# INSIGHT: Origem é Hit & Run
# INSIGHT: Destino são mulas descartáveis

## Provas e Falhas de Fraude por Lavagem

> **A Tese do Esvaziamento (Account Takeover)**: Indiferente da complexidade, a fraude financeira tende a seguir a regra universal do **Esvaziamento**. Em um cenário de _Account Takeover_ (Tomada de Conta), o criminoso assume o controle, transfere o saldo total para uma conta "laranja" (pivô) e realiza o saque imediatamente. O objetivo é ocultar a origem (Lavagem de Dinheiro) e não deixar rastro financeiro.

### Prova 1: Fluxo de Entrada

> Hipótese: A fraude só existe em dois momentos: na saída do dinheiro da vítima e no saque do criminoso. Roubo e saque.

In [0]:
# Contagem de fraude por tipo

df_fraud_query = df_final.groupBy("type") \
    .agg(F.sum("is_fraud").alias("total_fraudes")) \
    .withColumnRenamed("type", "tipo") \
    .orderBy("total_fraudes", ascending =  False)

display(df_fraud_query)

In [0]:
flow_proof = df_fraud_query.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))

max_valor = flow_proof['total_fraudes'].max()
ax.set_ylim(0, max_valor * 1.15) 

sns.barplot(
    data=flow_proof, 
    x="tipo", 
    y="total_fraudes", 
    color="#ff4d4d",
    ax=ax,
    edgecolor='none'
)

ax.bar_label(ax.containers[0], padding=8, fontweight='bold', fontsize=12)

plt.title("Prova de Fluxo: Acomplamento 1:1", fontsize=16, fontweight='bold', pad=20)

sns.despine(left=True, bottom=False) 

plt.xlabel("Tipo de Transação", fontsize=14, fontweight='bold', labelpad=15)
plt.ylabel("Total de Fraudes", fontsize=14, fontweight='bold', labelpad=15)

ax.tick_params(axis='x', pad=10)
plt.xticks(rotation=45, ha='right')  

plt.tight_layout(pad=2.0)
plt.show()


1. Existe um acoplamento quase perfeito **(1:1)**. Para cada Transferência (Roubo da Vítima), existe um Saque (Mula retirando o dinheiro). 
2. Tipos como `PAYMENT` (pagar boleto) ou `CASH_IN` (depósito) são ruído. Deve-se focar somente em `TRANSFER` e `CASH_OUT`.


In [0]:
# Df com somente linhas do tipo TRANSFER ou CASH_OUT.

df_focus = df_final.filter(F.col("type").isin(["TRANSFER", "CASH_OUT"]))

print("Foco definido em TRANSFER e CASH_OUT.")
display(df_focus.count())

### Prova 2: Padrão de Esvaziamento

> Hipótese: Se fizermos um gráfico nas instância onde há fraude confirmada do tipo transferência, o fraudador tenta levar o máximo possível


In [0]:
emptying_proof = df_focus \
  .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
  .select(F.col("old_balance_org"), F.col("amount")) \
  .toPandas()

fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=emptying_proof, 
    x='old_balance_org', 
    y='amount', 
    alpha=0.6,          # Transparência para ver onde há acúmulo de pontos
    color='#ff4d4d',    # O mesmo vermelho limpo do gráfico de barras
    edgecolor=None,
    ax=ax
)

# Linha de Referência

max_val = max(emptying_proof['old_balance_org'].max(), emptying_proof['amount'].max())

plt.plot([0, max_val], [0, max_val], 
         color='#333333',      # Cinza escuro elegante
         linestyle='--',
         linewidth=2, 
         label='Esvaziamento Total (Transferência = Saldo)')

plt.title("A Regra do Esvaziamento: Transferência de 100% do Saldo", fontsize=16, fontweight='bold', pad=20)
sns.despine() # Remove automaticamente a borda superior e a direita
plt.xlabel("Saldo Original da Vítima", fontsize=14, fontweight='bold', labelpad=15)
plt.ylabel("Valor Transferido", fontsize=14, fontweight='bold', labelpad=15)

ax.tick_params(axis='both', pad=10)

plt.legend(frameon=False, fontsize=12, loc='upper left')
plt.tight_layout(pad=2.0)
plt.show()


1. A linha perfeita de 45 graus confirma que, na esmagadora maioria dos casos, `Valor = Saldo`.
2. Além disso, notamos um teto horizontal em **10 Milhões**. Isso revela que, mesmo que a vítima tenha 60 milhões, o fraudador bate no limite transacional do sistema.

### Prova 3: O rastro das Contas Laranja (Mulas)

> Hipótese: Se é um roubo seguido de saque, a conta de destino deve começar zerada. Contas receptoras (mulas) são descartáveis e nascem zeradas para receber o ilícito.

In [0]:
mule_proof = df_focus \
    .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
    .withColumn(
        "tipo_mula",
        F.when((F.col("old_balance_dest") == 0) & (F.col("new_balance_dest") == F.col("amount")), "Laranja Perfeito (Entra e Fica)")
         .when((F.col("old_balance_dest") == 0) & (F.col("new_balance_dest") == 0), "Falha no Saldo (Entra e Some)")
         .otherwise("Conta com Saldo Prévio")
    )

df_summary = mule_proof.groupBy("tipo_mula").count()

In [0]:
pdf_summary = df_resumo.toPandas()

total_frauds = pdf_summary['count'].sum()
pdf_summary['percentile'] = (pdf_summary['count'] / total_frauds) * 100

pdf_summary = pdf_summary.sort_values('percentile', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#ff4d4d' if p > 50 else '#cccccc' for p in pdf_summary['percentile']]

sns.barplot(
    data=pdf_summary, 
    x='percentile', 
    y='tipo_mula', 
    hue='tipo_mula',
    palette=colors,
    legend=False,
    ax=ax,
    edgecolor='none'
)

for index, (porc, count) in enumerate(zip(pdf_summary['percentile'], pdf_summary['count'])):
    texto = f"{porc:.1f}% ({count:,})"
    ax.text(porc + 1, index, texto, va='center', fontweight='bold', fontsize=12)

plt.title("Prova das Contas Mulas", fontsize=16, fontweight='bold', pad=20, loc='center')

sns.despine(left=True, bottom=True)
plt.ylabel("")
ax.tick_params(axis='y', length=0, labelsize=12, pad=10)
ax.get_xaxis().set_visible(False)
ax.set_xlim(0, 115) 
plt.tight_layout(pad=2.0)
plt.show()


- **Laranja com "Erro" de Saldo (99.3%):** A conta estava zerada, recebeu o dinheiro, mas o log de `newBalanceDest` permaneceu zerado ou inconsistente.
- **Conta com Saldo Prévio (0.7%):** Contas que já tinham dinheiro (mistura de fundos).
- **Laranja Perfeito (0%):** Onde a matemática bate exata.

### Falha 1: A Rede de Lavagem

> Hipótese: A conta que recebe a transferência fraudulenta (name_dest em TRANSFER) é a mesma conta que realiza o saque logo em seguida (name_orig em CASH_OUT).

Resposta: Falha. O PaySim não mantém a consistência da identidade da mula ao longo do tempo (IDs reutilizados ou aleatórios).

- Há somente uma instância onde isso ocorre no conjunto inteiro.

In [0]:
df_input = df_focus \
    .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
    .select(
        F.col("step").alias("step_entrada"),
        F.col("name_dest").alias("conta_mula"),
        F.col("amount").alias("valor_entrada")
    )

df_output = df_focus \
    .filter(F.col("type") == "CASH_OUT") \
    .select(
        F.col("step").alias("step_saida"),
        F.col("name_orig").alias("conta_mula_saque"),
        F.col("amount").alias("valor_saida")
    )

# Realize o join em conta_mula com conta_mula_saque

df_cicle = df_input.join(
    df_output,
    on = df_input.conta_mula == df_output.conta_mula_saque,
    how ="inner"
)

df_cicle_timing = df_cicle \
    .filter(F.col("step_saida") >= F.col("step_entrada")) \
    .withColumn("tempo_para_lavar", F.col("step_saida") - F.col("step_entrada")) \
    .orderBy("tempo_para_lavar")

display(df_cicle_timing)


### Falha 2: O Horário do Crime

> Hipótese: Fraudes geralmente acontecem de madrugada.

Resposta: Falha. Volume absoluto de fraudes é constante nas 24h.

- Por demonstrar intensidade similar durante o dia inteiro, pode ser prova do uso de bots.

In [0]:
# df_hourpeak_query = total de fraudes por hora

df_hourpeak_query = df_final.groupBy("hour_of_day") \
    .agg(F.sum("is_fraud").alias("total_fraudes")) \
    .withColumnRenamed("hour_of_day", "hora") \
    .orderBy("total_fraudes", ascending =  False)

display(df_hourpeak_query.limit(5))

## FEATURE  - `ratio_amount_balance`

| Feature | O que é? |
| --- | --- |
| **`ratio_amount_balance`**    | Relação Valor / Saldo.              |

- 1.0 = Esvaziamento total.
- \>1.0 = Cheque especial/Bug (levou mais do que tinha) - Geralmente legítima.
- -1.0 = O saldo já era zero (erro/anomalia.)

In [0]:
# Criando a métrica `ratio_amount_balance'

df_focus = df_focus.withColumn("ratio_amount_balance", 
    F.when(F.col("old_balance_org") > 0, 
           F.col("amount") / F.col("old_balance_org")
    ).otherwise(-1.0) # Proteção contra divisão por zero
)

# Visualizando os casos de Esvaziamento Total (Ratio = 1)

display(df_focus.filter(F.col("ratio_amount_balance") == 1.0)
                .select("type", "amount", "old_balance_org", "ratio_amount_balance", "is_fraud")
                .limit(5))

%md
## FEATURES  - Ratio Amount

| Feature | O que é? |
| --- | --- |
| **`dest_is_empty`**            | Classificação textual ("Laranja Perfeito"). |
| **`type_idx`**                 | "TRANSFER" ou "CASH_OUT".                     |

In [0]:
# Transformação e criação de novas variáveis

df_transformed = df_focus \
    .withColumn("type_idx", 
                F.when(F.col("type") == "TRANSFER", 0) # 0 = TRANSFER
                 .otherwise(1)                         # 1 = CASH_OUT
    ) \
    .withColumn("dest_is_empty", 
                F.when(F.col("old_balance_dest") == 0, 1) # 1 = Mula Padrão
                 .otherwise(0)                            # 0 = Com Histórico
    )

## Consolidando o DataFrame

In [0]:
# Seleção das colunas relevantes para o treinamento

features_list = [
    "step",
    "hour_of_day",
    "type_idx",
    "amount",
    "old_balance_org",
    "error_orig",
    "old_balance_dest",
    "error_dest",
    "dest_is_empty",
    "ratio_amount_balance",
    "is_fraud"
]

df_ml_final = df_transformed.select(features_list)

In [0]:
df_ml_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("paysim_gold")